In [1]:
import pandas as pd
import time
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score

# Import all requested models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [2]:
df = pd.read_csv('../Dataset/WB_Dataset1_with_lon-lat.csv')

# Separate Features and Target
X = df.drop('crop_name', axis=1)
y = df['crop_name']

# Preprocessing: One-Hot Encoding
X = pd.get_dummies(X, columns=['soil_type', 'season'], drop_first=True)

# Preprocessing: Label Encoding for Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Preprocessing: Scaling
# (Required for KNN, SVM, and Linear models to work correctly)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split Data (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

In [3]:
models = {
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Linear (Logistic Reg)": LogisticRegression(max_iter=1000, random_state=42),
    "SVR (Implemented as SVC)": SVC(kernel='linear', random_state=42)
}

In [4]:
print(f"{'Model Name':<30} | {'Prediction Time (sec)':<25} | {'Accuracy':<10}")
print("-" * 75)

results = []

for name, model in models.items():
    # 1. Train the model
    model.fit(X_train, y_train)
    
    # 2. Measure Prediction Time
    start_time = time.time()
    try:
        if hasattr(model, "predict_proba"):
            probs = model.predict_proba(X_test)
            y_pred = np.argmax(probs, axis=1)
        else:
            # Fallback for Ridge/Linear models using decision_function
            probs = model.decision_function(X_test)
            y_pred = np.argmax(probs, axis=1)
            
        end_time = time.time()
        pred_time = end_time - start_time
        
        # Calculate Metrics
        acc_top1 = accuracy_score(y_test, y_pred)
        
        # Top-3 Logic
        if probs.shape[1] == len(le.classes_):
            top3_indices = np.argsort(probs, axis=1)[:, -3:]
            correct_top3 = sum([1 for i in range(len(y_test)) if y_test[i] in top3_indices[i]])
            acc_top3 = correct_top3 / len(y_test)
            acc_top3_str = f"{acc_top3*100:.2f}%"
        else:
            acc_top3_str = "N/A"
            
    except Exception as e:
        y_pred = model.predict(X_test) # Fallback
        acc_top1 = accuracy_score(y_test, y_pred)
        pred_time = 0
        acc_top3_str = "N/A"

    print(f"{name:<35} | {pred_time:.6f}   | {acc_top1*100:.2f}%     | {acc_top3_str:<10}")

Model Name                     | Prediction Time (sec)     | Accuracy  
---------------------------------------------------------------------------
Gradient Boosting                   | 0.021285   | 55.33%     | 87.33%    
Linear (Logistic Reg)               | 0.000733   | 57.33%     | 86.67%    
SVR (Implemented as SVC)            | 0.054971   | 56.67%     | 86.33%    
